In [1]:
# extract
import numpy as np
from numpy import linalg as LA

from tensorflow.keras.applications.resnet50  import ResNet50
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Model

class getResNet50Model:
    def __init__(self):
        self.input_shape = (224, 224, 3)
        self.resnet_model = ResNet50(weights='imagenet', input_shape=self.input_shape, include_top = True)
        self.output = self.resnet_model.get_layer('avg_pool').output
        self.resnet_model = Model(self.resnet_model.input, self.output)
        #self.resnet_model.summary()

    def extract_feat(self, img_path):
        img = image.load_img(img_path, target_size=(self.input_shape[0], self.input_shape[1]))
        img = image.img_to_array(img)
        img = np.expand_dims(img, axis=0)
        img = preprocess_input(img)
        feat = self.resnet_model.predict(img)
        norm_feat = feat[0]/LA.norm(feat[0])
        return norm_feat

In [2]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [3]:
import os
import h5py
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

images_path ="/content/gdrive/MyDrive/ml/train+vali/"
path = "/content/gdrive/MyDrive/ml/train+vali/"
model = getResNet50Model()

feats = []
names = []

for root, dirs, files in os.walk(images_path):
    for f in files:
      full_path = os.path.join(root, f)
      X = model.extract_feat(full_path)
      feats.append(X)
      rel_path = os.path.relpath(full_path, images_path)
      names.append(rel_path)
feats = np.array(feats)
output = "RestnetFeatures.h5"

h5f = h5py.File(output, 'w')
h5f.create_dataset('dataset_1', data = feats)
h5f.create_dataset('dataset_2', data = np.bytes_(names))
h5f.close()

102967424/102967424 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 354ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 232ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 238ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 255ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 231ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 214ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 205ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 218ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 227ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 235ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 207ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 270ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 362ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 382ms/step
1/1 ━━━━━━━━━━

In [6]:
# เปิดไฟล์ HDF5
with h5py.File('RestnetFeatures.h5', 'r') as h5f:
    # ตรวจสอบ keys ในไฟล์
    print("Keys in the HDF5 file:", list(h5f.keys()))

    # ตรวจสอบข้อมูลใน dataset_2 ก่อน
    dataset_2 = h5f['dataset_2']
    print(f"Dataset_2 info: {dataset_2}")

    # ถ้า dataset_2 เป็น scalar หรือเป็น array ที่ไม่สามารถ slice ได้
    if dataset_2.shape == ():
        # ถ้าเป็น scalar, จะดึงค่ามาเป็นค่าคงที่
        print("Dataset_2 is scalar, can't slice it.")
        names = dataset_2[()]
    else:
        # ถ้าเป็น array, สามารถ slice ได้
        names = dataset_2[:]

    print(f"Names of images (dataset_2): {names}")

    # ตัวอย่างข้อมูลแรกในแต่ละ dataset
    feats = h5f['dataset_1'][:]
    print(f"First feature vector: {feats[0]}")

Keys in the HDF5 file: ['dataset_1', 'dataset_2']
Dataset_2 info: <HDF5 dataset "dataset_2": shape (2954,), type "|S191">
Names of images (dataset_2): [b'Machu Pichu/10.jpg' b'Machu Pichu/17.jpg' b'Machu Pichu/13.jpg' ...
 b'Venezuela Angel Falls/383.jpg' b'Venezuela Angel Falls/389.jpg'
 b'Venezuela Angel Falls/403.jpg']
First feature vector: [0.05504626 0.01173174 0.0296077  ... 0.00487872 0.00138772 0.00283495]
